In [78]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import json, random
from datasets import Dataset
import os

In [80]:

# 一些参数
model_path = '/root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-0___5B-Instruct'
# model_name = "/qwen2.5-0.5-instruct/"  # 模型名或者本地路径
train_data_path = '../../dataset/prep/1/train.json'
test_file_path = '../../dataset/prep/1/test.json'
res_path = '../../dataset/res/tmp_res1.json'
save_path = "./output/save_model_7b/"

num_train_epochs=2
per_device_train_batch_size=2
per_device_eval_batch_size=1
warmup_steps=10
weight_decay=0.01
logging_steps=1
use_cpu=False

# 创建标签到索引的映射
label_to_id = {
    "胸痹心痛病": 0,
    "心衰病": 1,
    "眩晕病": 2,
    "心悸病": 3
}

num_labels = len(label_to_id)  # 根据你的标签数量设置num_labels

with open(train_data_path, 'r', encoding='utf-8') as file:
    # 使用 json.load() 方法将文件内容解析为 Python 对象
    data = json.load(file)

random.shuffle(data)



# 将文本标签转换为数值标签
for example in data:
    example['label'] = label_to_id[example['output']]

# 检查标签范围
for example in data:
    assert 0 <= example['label'] < len(label_to_id), f"Label out of range: {example['output']}" 

train_data = []
for d in data:
    tmp = {}
    tmp['input'] = d['input']
    tmp['label'] = d['label']
    train_data.append(tmp)

# 将数据转换为datasets库的Dataset对象
dataset = Dataset.from_list(train_data)

# 将数据集拆分为训练集和验证集
dataset = dataset.train_test_split(test_size=0.2)

In [82]:
# 加载预训练的 Qwen2 模型和分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=num_labels)  
print(model)
model.config.pad_token_id = 151643  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at /root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-0___5B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Qwen2ForSequenceClassification(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): 

In [83]:



# 定义一个函数来处理数据集中的文本
def preprocess_function(examples):
    return tokenizer(examples['input'], truncation=True, padding=True, return_tensors="pt")

# 对数据集进行预处理
encoded_dataset = dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

In [84]:

# 定义训练参数
training_args = TrainingArguments(
    output_dir=save_path,                           # 输出目录
    num_train_epochs=num_train_epochs,              # 训练的epoch数
    per_device_train_batch_size=per_device_train_batch_size,    # 每个设备的训练batch size
    per_device_eval_batch_size=per_device_eval_batch_size,      # 每个设备的评估batch size
    warmup_steps=warmup_steps,                  # 预热步数
    weight_decay=weight_decay,                  # 权重衰减
    logging_dir=save_path,                      # 日志目录
    logging_steps=logging_steps,
    evaluation_strategy="epoch",
    save_strategy="epoch",    # 每个epoch保存一次检查点
    save_total_limit=3,       # 最多保存3个检查点，旧的会被删除
    use_cpu=False
)

# 定义Trainer
trainer = Trainer(
    model=model,                                    # 模型
    args=training_args,                             # 训练参数
    train_dataset=encoded_dataset['train'],         # 训练数据集
    eval_dataset=encoded_dataset['test']            # 评估数据集
)

# 打印训练集和验证集中的一些样本
print("Train dataset sample:")
print(encoded_dataset['train'][0])  # 打印训练集中的第一个样本

print("Eval dataset sample:")
print(encoded_dataset['test'][0])  # 打印验证集中的第一个样本

# 开始训练
trainer.train()
trainer.save_state()
trainer.save_model(output_dir=save_path)
tokenizer.save_pretrained(save_path)

/root/miniconda3/lib/python3.10/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Train dataset sample:
{'input': '\n[症状]:阵发性头晕，视物不清，自感天旋地转，走路歪斜，下肢无力，伴阵发性心慌，左侧足趾疼痛，神志清，精神可，无恶心呕吐，无发热咳嗽，纳可，眠易醒，醒后难以再入睡，小便调，大便干，4-5日一行。\n[中医望闻切诊]:中医望闻切诊：表情自然，面色少华，形体正常，望态，语气清，气息平，无异常气味，舌暗红、苔白腻,脉代。\n', 'label': 2, 'input_ids': [198, 58, 101368, 5669, 99854, 28291, 33071, 116280, 3837, 57452, 52853, 106122, 3837, 35926, 98650, 35727, 100917, 29490, 46670, 3837, 109214, 107687, 102669, 3837, 16872, 101775, 107856, 3837, 99595, 99854, 28291, 33071, 63109, 102838, 3837, 111687, 99336, 115702, 105748, 3837, 99315, 77128, 79766, 3837, 100150, 30440, 3837, 42192, 110887, 113052, 3837, 42192, 108976, 109244, 3837, 99458, 30440, 3837, 101519, 86744, 99951, 3837, 99951, 33447, 104151, 87256, 117336, 3837, 30709, 99364, 47872, 3837, 26288, 99364, 99251, 3837, 19, 12, 20, 8903, 105643, 8997, 58, 104823, 99317, 99608, 99322, 99781, 5669, 104823, 99317, 99608, 99322, 99781, 5122, 102936, 99795, 3837, 113906, 82647, 85361, 3837, 82699, 31914, 100416, 3837, 99317, 35243, 3837, 110098, 79766, 3837, 103007,

Epoch,Training Loss,Validation Loss
1,2.519400,1.182553
2,0.000900,1.709799


('./output/save_model_7b/tokenizer_config.json',
 './output/save_model_7b/special_tokens_map.json',
 './output/save_model_7b/vocab.json',
 './output/save_model_7b/merges.txt',
 './output/save_model_7b/added_tokens.json',
 './output/save_model_7b/tokenizer.json')

# 测试

In [86]:
from transformers import Qwen2ForSequenceClassification, Qwen2Tokenizer
import torch
import json

# 加载模型和分词器
tokenizer = Qwen2Tokenizer.from_pretrained(save_path)
model = Qwen2ForSequenceClassification.from_pretrained(save_path)
for parameter in model.parameters():
    parameter.requires_grad = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Qwen2ForSequenceClassification(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896, padding_idx=151643)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      

In [87]:
# 创建标签到索引的映射
label_to_id = {
    "胸痹心痛病": 0,
    "心衰病": 1,
    "眩晕病": 2,
    "心悸病": 3
}
# 创建 id_to_label 字典
id_to_label = {v: k for k, v in label_to_id.items()}


with open(test_file_path, 'r', encoding='utf-8') as file:
    # 使用 json.load() 方法将文件内容解析为 Python 对象
    data = json.load(file)
    
# 准备输入文本
texts = []
true_label = []
ID = []

for line in data[:50]:
    t = line['input']
    texts.append(t)
    # true_label.append(label_to_id[line['疾病']])
    ID.append(line['ID'])


In [88]:

# 对文本进行编码
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
# 将输入数据移动到指定设备
inputs = {k: v.to(device) for k, v in inputs.items()}


In [89]:
# 进行推理
with torch.no_grad():
    outputs = model(**inputs)

# 获取预测结果
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1)

results = []
for num, i in enumerate(predictions):
    results.append({"ID": ID[num], "疾病": id_to_label[i.item()]})

with open(res_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)